# 6 — Customising the Characterisation Matrix — Solutions
Worked solutions for all exercises in notebook 6.

## Setup

In [ ]:
import bw2data as bd
import bw2calc as bc
import numpy as np
import scipy.sparse as sp

bd.projects.set_current('BAFU 2025 v2')
db = bd.Database('BAFU-2025-v2')
print(f'Database: {db.name}, {len(db)} processes')

node_location = {n.id: n.get('location', 'GLO') for n in db}
print(f'Location cache: {len(node_location)} entries')

## Exercise 1 — Scandinavian 2× characterisation factor

Danish electricity has about 25% of its GW supply chain in Scandinavia (natural gas from Norway, Swedish electricity imports). Swiss certified electricity has essentially none.

**Approach:** build a characterisation matrix where `char_matrix[i, j] = 2 × CF_i` for Scandinavian process columns, `CF_i` elsewhere. Then `characterized_inventory = char_matrix.multiply(lca.inventory)` and `float(characterized_inventory.sum())` gives the score.

In [ ]:
dk_electricity = bd.get_node(
    database='BAFU-2025-v2',
    name='Electricity, low voltage, production DK, at grid',
    location='DK',
)
ch_electricity = bd.get_node(
    database='BAFU-2025-v2',
    name='Electricity, low voltage, certified electricity, at grid',
    location='CH',
)

SCANDINAVIAN_LOCATIONS = {'NO', 'SE', 'DK', 'FI'}


def score_with_scand_multiplier(node, multiplier=2.0):
    functional_unit, data_objs, _ = bd.prepare_lca_inputs(
        demand={node: 1},
        method=('Ecological Scarcity 2021', 'Global warming'),
        remapping=False,
    )
    lca = bc.LCA(demand=functional_unit, data_objs=data_objs)
    lca.lci()
    lca.lcia()
    standard = lca.score

    cf_vector = np.array(lca.characterization_matrix.sum(axis=1)).ravel()
    inv_coo = lca.inventory.tocoo()

    # Map column index → location
    col_to_loc = {col: node_location.get(nid, 'GLO')
                  for nid, col in lca.dicts.activity.items()}

    # Start with standard CFs; double for Scandinavian process columns
    cf_values = cf_vector[inv_coo.row].copy()
    is_scand = np.array([col_to_loc.get(col, 'GLO') in SCANDINAVIAN_LOCATIONS
                         for col in inv_coo.col])
    cf_values[is_scand] *= multiplier

    char_matrix = sp.coo_matrix(
        (cf_values, (inv_coo.row, inv_coo.col)),
        shape=lca.inventory.shape
    )
    characterized_inventory = char_matrix.multiply(lca.inventory)
    modified = float(characterized_inventory.sum())

    # Scandinavian contribution at standard factor (for reporting)
    cf_scand = cf_vector[inv_coo.row].copy()
    cf_scand[~is_scand] = 0.0
    scand_char = sp.coo_matrix((cf_scand, (inv_coo.row, inv_coo.col)), shape=lca.inventory.shape)
    scand_contribution = float(scand_char.multiply(lca.inventory).sum())

    return standard, modified, scand_contribution


for node, label in [(dk_electricity, 'Danish electricity'), (ch_electricity, 'Swiss electricity')]:
    std, mod, scand_cont = score_with_scand_multiplier(node)
    change_pct = 100 * (mod - std) / std
    print(f'{label} ({node["location"]}):')
    print(f'  Standard score:     {std:.2f} UBP')
    print(f'  Scand contribution: {scand_cont:.2f} UBP ({100*scand_cont/std:.0f}%)')
    print(f'  With 2x Scand:      {mod:.2f} UBP ({change_pct:+.0f}%)')
    print()

## Exercise 2 — Wind and solar reduce impacts

The wind farm process has GW emissions from turbine manufacturing. Giving a 90% discount to all renewable electricity process columns means those columns contribute only 10% of their standard characterised score.

**Approach:** build a characterisation matrix where `char_matrix[i, j] = 0.1 × CF_i` for renewable electricity process columns, `CF_i` elsewhere.

In [ ]:
wind_farm = bd.get_node(
    database='BAFU-2025-v2',
    name='Electricity, at 100MW wind farm, onshore, 2.3MW turbines',
    location='RER',
)
gas_se = bd.get_node(
    database='BAFU-2025-v2',
    name='Electricity, natural gas, at CHP power plant {SE} U - SE',
    location='SE',
)

renewable_ids = {
    n.id for n in db
    if (
        ('wind farm' in n['name'].lower() and 'electricity' in n['name'].lower())
        or 'production mix photovoltaic' in n['name'].lower()
    )
    and not n['name'].startswith('xx')
}
print(f'Renewable electricity processes found: {len(renewable_ids)}')


def score_with_renewable_discount(node, discount_factor=0.1):
    functional_unit, data_objs, _ = bd.prepare_lca_inputs(
        demand={node: 1},
        method=('Ecological Scarcity 2021', 'Global warming'),
        remapping=False,
    )
    lca = bc.LCA(demand=functional_unit, data_objs=data_objs)
    lca.lci()
    lca.lcia()
    standard = lca.score

    cf_vector = np.array(lca.characterization_matrix.sum(axis=1)).ravel()
    inv_coo = lca.inventory.tocoo()

    # Column indices that are renewable in this LCA's supply chain
    renew_cols = {lca.dicts.activity[nid] for nid in renewable_ids if nid in lca.dicts.activity}

    cf_values = cf_vector[inv_coo.row].copy()
    is_renew = np.array([col in renew_cols for col in inv_coo.col])
    cf_values[is_renew] *= discount_factor

    char_matrix = sp.coo_matrix(
        (cf_values, (inv_coo.row, inv_coo.col)),
        shape=lca.inventory.shape
    )
    characterized_inventory = char_matrix.multiply(lca.inventory)
    modified = float(characterized_inventory.sum())

    # Renewable contribution at standard factor (for reporting)
    cf_renew = cf_vector[inv_coo.row].copy()
    cf_renew[~is_renew] = 0.0
    renew_char = sp.coo_matrix((cf_renew, (inv_coo.row, inv_coo.col)), shape=lca.inventory.shape)
    renew_contribution = float(renew_char.multiply(lca.inventory).sum())

    return standard, modified, renew_contribution, len(renew_cols)


for node, label in [(wind_farm, 'Wind electricity'), (gas_se, 'Gas CHP (SE)')]:
    std, mod, renew_cont, n_renew = score_with_renewable_discount(node)
    change_pct = 100 * (mod - std) / std
    print(f'{label}:')
    print(f'  Renewable cols in supply chain: {n_renew}')
    print(f'  Renewable contribution:         {renew_cont:.4f} UBP ({100*renew_cont/std:.2f}%)')
    print(f'  Standard score:    {std:.4f} UBP')
    print(f'  With 90% discount: {mod:.4f} UBP ({change_pct:+.2f}%)')
    print()

## Geometric land use — full worked code

For each non-zero $(i, j)$ in the land use inventory, `char_matrix[i, j] = CF_{\text{base}} \times L_c^{\alpha-1}` where $c$ is the country of process $j$.

In [ ]:
method_lu = bd.Method(('Ecological Scarcity 2021', 'Land use'))
lu_flow_ids = {key for key, _ in method_lu.load()}

rape_ch = bd.get_node(database='BAFU-2025-v2', name='Rape seed IP, at farm', location='CH')
functional_unit, data_objs, _ = bd.prepare_lca_inputs(
    demand={rape_ch: 1},
    method=('Ecological Scarcity 2021', 'Land use'),
    remapping=False,
)
lca_lu = bc.LCA(demand=functional_unit, data_objs=data_objs)
lca_lu.lci()
lca_lu.lcia()

lu_rows = [lca_lu.dicts.biosphere[k] for k in lu_flow_ids if k in lca_lu.dicts.biosphere]
raw_lu_per_process = np.array(lca_lu.inventory[lu_rows, :].sum(axis=0)).ravel()

# Aggregate land use by country using location cache
country_lu = {}
for col_idx in np.where(raw_lu_per_process > 1e-9)[0]:
    node_id = lca_lu.dicts.activity.reversed[col_idx]
    country = node_location.get(node_id, 'GLO')
    country_lu[country] = country_lu.get(country, 0) + raw_lu_per_process[col_idx]

total_lu = sum(country_lu.values())
CF_annual_crop = 520
alpha = 1.5

# Column index → location
col_to_loc = {col: node_location.get(nid, 'GLO')
              for nid, col in lca_lu.dicts.activity.items()}

# Build COO and assign geometric CFs
inv_coo = lca_lu.inventory.tocoo()

cf_base = np.zeros(lca_lu.inventory.shape[0])
for row in lu_rows:
    cf_base[row] = CF_annual_crop

L_c_per_nz = np.array([country_lu.get(col_to_loc.get(col, 'GLO'), 0.0) for col in inv_coo.col])
geo_factor = np.where(L_c_per_nz > 0, L_c_per_nz ** (alpha - 1), 1.0)

char_matrix_lu = sp.coo_matrix(
    (cf_base[inv_coo.row] * geo_factor, (inv_coo.row, inv_coo.col)),
    shape=lca_lu.inventory.shape
)
characterized_inventory = char_matrix_lu.multiply(lca_lu.inventory)
geometric_score = float(characterized_inventory.sum())
standard_approx = total_lu * CF_annual_crop

# Hypothetical distributed product
n_c = 4
geometric_dist = n_c * ((total_lu / n_c) ** alpha) * CF_annual_crop

print(f"{'Product':<35} {'Standard':>10} {'Geometric (a=1.5)':>18}")
print('-' * 65)
print(f"{'Concentrated (2.92 m2.yr in CH)':<35} {standard_approx:>10.0f} {geometric_score:>18.0f}")
print(f"{'Distributed (0.73 in each of 4)':<35} {standard_approx:>10.0f} {geometric_dist:>18.0f}")
print()
print(f'Standard LCA: both products score identically ({standard_approx:.0f} UBP)')
print(f'Geometric LCA: concentrated is {geometric_score/geometric_dist:.1f}x worse than distributed')